# Tissue elevation heatmap -- investigation

Prototypes a per-pixel "elevation" map of tissue z-extent (a digital-
elevation-model-style heatmap, not the single scalar-per-FOV thickness
`after_imaging/08_measure_tissue_thickness.ipynb` already computes), plus a
z-sweep GIF of the downsampled, flat-field-corrected DAPI signal. This
notebook only investigates the approach on real data -- new reusable
functions stay local here; wiring anything into `MERci` proper (or into
`08_measure_tissue_thickness.ipynb`) is a follow-up session's task.

**Real test dataset**: `LT066_sample_01/merfish` (1138 real imaged FOVs,
101-z-step DAPI (405 nm) `cells` round, ST2 microscope) -- same dataset
`notebooks/tests/decrease_fov_number/01_*`/`02_*` already investigate, same
`SAMPLE_DIR`-explicit convention (too large to copy locally; read directly,
cache only computed/downsampled results).

**Algorithm** (see `prompt_history/` for the original request):
1. Identify **boundary FOVs** = the *exterior* FOVs of the imaged grid
   (`MERci.acquisition.positions.find_exterior_fovs` -- the same set
   `analysis/ffc.py`'s default `"exterior_grid"` strategy already selects
   for flat-field correction, since exterior FOVs are the ones assumed to be
   mostly tissue-free background).
2. Build the flat-field-correction (FFC) field from those FOVs, and estimate
   a background/foreground intensity threshold from them (same "highest
   pixel value among the N lowest-mean frames" estimator
   `misc/measure_tissue_thickness_test.ipynb`/`analysis/fov.compute_tissue_fraction`
   already use), computed in the same FFC-corrected + downsampled space the
   per-FOV elevation loop actually thresholds in.
3. For a **small representative block of FOVs** (not the full 1138-FOV grid
   -- see the scope note below), and for every z-plane: FFC-correct,
   downsample (factor 8: 2304 -> 288 px), threshold, and record the
   elevation matrix `M` (the topmost z, in µm, at which each downsampled
   pixel is still foreground).
4. Crop each FOV's `M` to its non-overlap footprint and stitch the block's
   FOVs into one grid-indexed elevation heatmap.
5. Stitch the same block's downsampled DAPI images per z into a GIF.

**Scope note**: this real experiment's `cells` round is ~572 MB/FOV x 1138
FOVs (~650 GB); a full-grid run of step 3 is a cluster/SLURM-array job for
another session (`misc/measure_tissue_thickness_test.ipynb` sections
14/23/24 already have that SLURM-array pattern for similarly heavy per-FOV
z-stack loops). Step 1-2 (boundary-FOV identification + FFC + threshold) DO
run over the real, full 1138-FOV grid below -- only a single representative
frame per boundary FOV is read for that (256 FOVs x 1 frame, ~2.7 GB), which
is cheap. Step 3 onward runs on one ~5x5 representative block only (chosen
to straddle a real tissue-boundary/hole edge, so it exercises both the FFC
boundary-FOV logic and the crop/stitch-across-neighbors logic).

## 1 -- Setup

In [ ]:
%matplotlib inline
# %matplotlib widget  # uncomment for interactive pan/zoom (ipympl)

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from skimage.measure import block_reduce
from PIL import Image, ImageDraw, ImageFont

# notebooks/tests/<subfolder>/ is three levels under the repo root (MERci/),
# same convention as notebooks/before_imaging/regular/ (3 levels).
MERCI_DIR = Path(os.getcwd()).parent.parent.parent
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config import ExperimentConfig
from MERci.common.metadata import ExperimentMetadata
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.common.io import read_image_frames
from MERci.acquisition.configs import get_fov_geometry, get_color_frame_indices
from MERci.acquisition.positions import find_exterior_fovs
from MERci.analysis.ffc import (
    select_ffc_exterior_fovs, compute_ffc_field_for_color, compute_mosaic_crop_px,
    save_ffc_field, load_ffc_field,
)
from MERci.progress_display import ProgressReporter
from MERci.visualization import get_merci_figures_dir

NOTEBOOK_NAME = "elevation_heatmap"

# NOTEBOOK_GUIDELINES.md #5 -- explicit plot font sizes, reused by every
# plotting cell below instead of matplotlib's figsize-relative defaults.
PLOT_TITLE_FONTSIZE, PLOT_LABEL_FONTSIZE = 13, 11
PLOT_TICK_FONTSIZE, PLOT_LEGEND_FONTSIZE = 10, 10

## 2 -- Parameters

In [ ]:
# Same real dataset as notebooks/tests/decrease_fov_number/01_*/02_* -- see
# those notebooks' own Parameters cells for why SAMPLE_DIR is set explicitly
# (too large to copy locally) rather than auto-detected from this repo's
# own location.
SAMPLE_DIR = Path("/n/holylfs05/LABS/zhuang_lab/Lab/shared/projects/lineage_tracing/experiments/LT066_sample_01/merfish")

MICROSCOPE = "ST2"
OBJECTIVE  = "60X"
CHANNEL_NM = 405.0   # DAPI

DOWNSAMPLE_FACTOR = 8   # 2304 / 8 = 288 px

# THRESHOLD estimation (same convention as misc/measure_tissue_thickness_test.ipynb
# section 5 / analysis.fov.compute_tissue_fraction): the highest pixel value
# observed among the N_BACKGROUND_FRAMES lowest-mean boundary-FOV frames, in
# the SAME FFC-corrected + downsampled space the per-FOV elevation loop
# actually thresholds in (see section 3 below).
N_BACKGROUND_FRAMES = 20

# Representative block for the per-FOV elevation-matrix step (see the
# Scope note above) -- a compact grid_rows x grid_cols window of the real
# FOV grid, searched for one that both meets MIN_BLOCK_FOVS and contains a
# boundary-FOV count in EXT_COUNT_RANGE (i.e. straddles a real tissue edge
# rather than sitting purely in the interior or purely on the perimeter).
BLOCK_GRID_ROWS  = 5
BLOCK_GRID_COLS  = 5
MIN_BLOCK_FOVS   = 15
EXT_COUNT_RANGE  = (3, 10)

# z-sweep GIF (same idiom as misc/measure_tissue_thickness_test.ipynb section 24)
GIF_Z_STRIDE          = 5     # every Nth z-step
GIF_FRAME_DURATION_MS = 300
GIF_DOWNSCALE_WIDTH_PX = 700

CACHE_DIR = SAMPLE_DIR / "analysis" / "cache" / NOTEBOOK_NAME
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR = get_merci_figures_dir(SAMPLE_DIR, "tests", NOTEBOOK_NAME, subfolder="tissue_thickness")

# The real experiment's own identity (via its deployed MERci/ clone under
# SAMPLE_DIR), NOT this repo's own location -- this notebook is not deployed
# inside SAMPLE_DIR (see the Scope note above), so MERCI_DIR resolves this
# repo's own identity instead, same distinction
# notebooks/tests/decrease_fov_number/02_*.ipynb already makes.
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(SAMPLE_DIR / "MERci")
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)

pixel_size_um, image_size_px = get_fov_geometry(MICROSCOPE, OBJECTIVE)

config = ExperimentConfig(
    data_dir=SAMPLE_DIR / "data", metadata_dir=SAMPLE_DIR / "metadata",
    analysis_dir=SAMPLE_DIR / "analysis", settings_dir=SAMPLE_DIR / "settings",
    round_info_csv=SAMPLE_DIR / "metadata" / "round_info.csv",
    positions_txt=SAMPLE_DIR / "positions" / f"positions_{POSITIONS_TAG}.txt",
    image_suffix=".zarr", microscope=MICROSCOPE,
    pixel_size_um=pixel_size_um, image_size_px=image_size_px,
)
meta = ExperimentMetadata.load(config.round_info_csv, config.positions_txt, config.data_dir,
                                image_suffix=config.image_suffix)

def resolve_round_id(meta, imaging_type):
    for rid in meta.valid_round_ids():
        if any((s.imaging_type or "").strip().lower() == imaging_type for s in meta.series_for_round(rid)):
            return rid
    raise ValueError(f"No round found with imaging_type={imaging_type!r}")

cells_round_id = resolve_round_id(meta, "cells")
round_info = meta.rounds[cells_round_id]

# {fov_id: (x, y)} -- scoped to THIS round's own real imaged FOVs (matches
# select_ffc_exterior_fovs's own internal restriction), not the raw
# experiment-wide positions.txt, so transit-only FOVs never enter the set.
positions = {fov_id: meta.fovs[fov_id].position
             for fov_id in round_info.fov_files if round_info.fov_files[fov_id]}

for s in meta.series_for_round(cells_round_id):
    if s.hal_config:
        from MERci.acquisition.configs import find_frame_table_for_hal_config
        ft_path = find_frame_table_for_hal_config(config.settings_dir / s.hal_config, config.metadata_dir)
        break
frame_table = pd.read_csv(ft_path, index_col=0)

channel_frames  = frame_table[frame_table["color"].round(0) == round(CHANNEL_NM)].sort_values("z")
z_frame_indices = channel_frames.index.tolist()
z_um_values     = channel_frames["z"].tolist()
mid_frame_idx   = get_color_frame_indices(frame_table)[CHANNEL_NM]

print(f"SAMPLE_DIR   : {SAMPLE_DIR}")
print(f"cells round  : {cells_round_id}  ({len(positions)} FOV(s) with real files)")
print(f"step_size_um : {config.step_size_um:.2f}   image_size_px: {config.image_size_px}")
print(f"{CHANNEL_NM} nm: {len(z_frame_indices)} z-plane(s), {z_um_values[0]:.1f}-{z_um_values[-1]:.1f} um, mid-z frame_idx={mid_frame_idx}")
print(f"Cache  : {CACHE_DIR}")
print(f"Figures: {FIGURES_DIR}")

## 3 -- Identify boundary FOVs

"Boundary FOVs" = the *exterior* FOVs of the imaged grid (outer perimeter +
any hole edges) -- `find_exterior_fovs`, the same definition
`analysis/ffc.py`'s `"exterior_grid"` FFC-candidate strategy already uses,
since exterior FOVs are the ones assumed to be mostly tissue-free
background.

In [ ]:
boundary_fov_ids = find_exterior_fovs(
    positions, config.step_size_um,
    connectivity=config.ffc_connectivity, tolerance_fraction=config.ffc_neighbor_tolerance,
)
print(f"{len(boundary_fov_ids)} / {len(positions)} FOVs are boundary (exterior-grid) FOVs")

grid_indices = {}
xy = np.array([positions[i] for i in positions])
x0, y0 = xy[:, 0].min(), xy[:, 1].min()
for fov_id, (x, y) in positions.items():
    col = int(round((x - x0) / config.step_size_um))
    row = int(round((y - y0) / config.step_size_um))
    grid_indices[fov_id] = (row, col)

## 4 -- Plot: which FOVs will be used for the FFC

Same plot idiom as `02_create_positions_from_boundaries.ipynb`'s "FOV
layout" plot (one rectangle per FOV footprint at its real stage position) --
boundary (FFC-candidate) FOVs highlighted, everything else in gray.

In [ ]:
half = config.step_size_um / config.non_overlap_fraction / 2   # true FOV footprint half-width

fig, ax = plt.subplots(figsize=(8, 7))
for fov_id, (x, y) in positions.items():
    is_boundary = fov_id in boundary_fov_ids
    ax.add_patch(mpatches.Rectangle(
        (x - half, y - half), 2 * half, 2 * half,
        lw=0.3, edgecolor="tab:red" if is_boundary else "0.6",
        facecolor="tab:red" if is_boundary else "0.6",
        alpha=0.6 if is_boundary else 0.15,
    ))
ax.plot([], [], "s", color="tab:red", alpha=0.6, ms=8, label=f"boundary / FFC ({len(boundary_fov_ids)})")
ax.plot([], [], "s", color="0.6", alpha=0.3, ms=8, label=f"interior ({len(positions) - len(boundary_fov_ids)})")
ax.invert_yaxis(); ax.axis("equal")
ax.set_title(f"FOVs used for FFC -- {SAMPLE_NAME}", fontsize=PLOT_TITLE_FONTSIZE)
ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
ax.legend(fontsize=PLOT_LEGEND_FONTSIZE, loc="best")
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.boundary_fovs.png", dpi=150)
plt.show()

## 5 -- FFC field from boundary FOVs

One mid-z frame per boundary FOV (`select_ffc_exterior_fovs`) feeds
`compute_ffc_field_for_color` -- cached under `CACHE_DIR` (kept separate
from the production `ProgressTracker`-managed FFC cache under
`analysis/ffc_fields/`, since this is an investigation-only field).

In [ ]:
ffc_cache_path = CACHE_DIR / f"ffc_field_{int(CHANNEL_NM)}nm.npz"

if ffc_cache_path.exists():
    ffc_field, ffc_meta = load_ffc_field(ffc_cache_path)
    print(f"Loaded cached FFC field: {ffc_cache_path}  ({ffc_meta})")
else:
    boundary_samples = select_ffc_exterior_fovs(cells_round_id, config, meta, mid_frame_idx)
    print(f"Computing FFC field from {len(boundary_samples)} boundary-FOV frame(s) ...")
    ffc_field, ffc_meta = compute_ffc_field_for_color(
        boundary_samples, smooth_sigma_px=config.ffc_smooth_sigma_px,
        normalize_percentile=config.ffc_normalize_percentile, ffc_min_value=config.ffc_min_value,
    )
    save_ffc_field(ffc_cache_path, ffc_field, ffc_meta)
    print(f"Saved: {ffc_cache_path}  ({ffc_meta})")

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
im = ax.imshow(ffc_field, cmap="viridis")
ax.set_title(f"FFC field ({CHANNEL_NM:.0f} nm)", fontsize=PLOT_TITLE_FONTSIZE)
ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
cbar = fig.colorbar(im, ax=ax); cbar.ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.ffc_field.png", dpi=150)
plt.show()

## 6 -- Boundary-FOV DAPI intensity overlay + threshold

Same idiom as `acquisition.mosaic.plot_tile_intensity_histograms`
(re-used, per that function's own docstring, by
`misc/measure_tissue_thickness_test.ipynb` section 5): every boundary FOV's
own log-space histogram as a thin line, plus a bold pooled histogram.
Computed in the **FFC-corrected + downsampled** space (not raw pixels), so
the resulting `THRESHOLD` applies directly to what section 8 below actually
thresholds -- each boundary FOV's same mid-z frame is FFC-corrected and
downsampled by `DOWNSAMPLE_FACTOR` exactly like every z-plane is in the
per-FOV elevation loop.

In [ ]:
def ffc_correct_and_downsample(frame, ffc_field, factor):
    corrected = np.clip(frame.astype(np.float64) / ffc_field, 0, None)
    return block_reduce(corrected, (factor, factor), func=np.mean)

boundary_ids_sorted = sorted(boundary_fov_ids)
boundary_ds_path = CACHE_DIR / f"boundary_fov_downsampled_{int(CHANNEL_NM)}nm.npz"

if boundary_ds_path.exists():
    _npz = np.load(boundary_ds_path)
    boundary_ds = {int(k.split("_")[1]): _npz[k] for k in _npz.files}
    print(f"Loaded {len(boundary_ds)} cached boundary-FOV downsampled frame(s): {boundary_ds_path}")
else:
    boundary_ds = {}
    reporter = ProgressReporter(total=len(boundary_ids_sorted), label="Reading+correcting boundary FOVs")
    for fov_id in reporter.wrap(boundary_ids_sorted):
        fpath = round_info.fov_files[fov_id][0]
        frame = read_image_frames(fpath, [mid_frame_idx])[0]
        boundary_ds[fov_id] = ffc_correct_and_downsample(frame, ffc_field, DOWNSAMPLE_FACTOR).astype(np.float32)
    np.savez_compressed(boundary_ds_path, **{f"fov_{k}": v for k, v in boundary_ds.items()})
    print(f"Saved: {boundary_ds_path}")

means = {fov_id: float(ds.mean()) for fov_id, ds in boundary_ds.items()}
lowest_mean_ids = sorted(means, key=means.get)[:N_BACKGROUND_FRAMES]
THRESHOLD = max(float(boundary_ds[fov_id].max()) for fov_id in lowest_mean_ids)
print(f"THRESHOLD = {THRESHOLD:.1f} (max pixel value among the "
      f"{N_BACKGROUND_FRAMES} lowest-mean boundary FOVs, FFC-corrected + downsampled space)")

In [ ]:
LOG_BINS = 200
all_vals = np.concatenate([np.clip(ds, 1, None).ravel() for ds in boundary_ds.values()])
bin_edges = np.linspace(np.log10(all_vals.min()), np.log10(all_vals.max()), LOG_BINS + 1)

fig, ax = plt.subplots(figsize=(8, 5))
pooled_hist = np.zeros(LOG_BINS)
for fov_id, ds in boundary_ds.items():
    hist, _ = np.histogram(np.log10(np.clip(ds, 1, None)), bins=bin_edges, density=True)
    ax.plot(bin_edges[:-1], hist, "-", lw=0.6, color="0.6", alpha=0.5)
    pooled_hist += hist
pooled_hist /= len(boundary_ds)
ax.plot(bin_edges[:-1], pooled_hist, "-", lw=2, color="k", label="pooled (mean over boundary FOVs)")
ax.axvline(np.log10(max(THRESHOLD, 1)), color="tab:red", ls="--",
           label=f"THRESHOLD = {THRESHOLD:.0f}")
ax.set_xlabel("log10(intensity)  [FFC-corrected, downsampled]", fontsize=PLOT_LABEL_FONTSIZE)
ax.set_ylabel("density", fontsize=PLOT_LABEL_FONTSIZE)
ax.set_title(f"Boundary-FOV DAPI intensity overlay ({len(boundary_ds)} FOVs)", fontsize=PLOT_TITLE_FONTSIZE)
ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
ax.legend(fontsize=PLOT_LEGEND_FONTSIZE)
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.boundary_fov_histograms.png", dpi=150)
plt.show()

## 7 -- Pick a representative FOV block

Per this notebook's scope note (section intro), the per-z elevation loop
(section 8) runs on one compact block of the real grid rather than the full
1138-FOV grid. The block is searched for automatically: the smallest
`BLOCK_GRID_ROWS x BLOCK_GRID_COLS` window (row-major scan) with at least
`MIN_BLOCK_FOVS` real FOVs and a boundary-FOV count inside
`EXT_COUNT_RANGE` -- i.e. one that straddles a real tissue edge (mix of
boundary + interior FOVs), rather than sitting purely in the interior or
purely on the perimeter, so it exercises both the FFC boundary-FOV logic
and the crop/stitch-across-a-real-edge logic.

In [ ]:
def pick_compact_block(grid_indices, boundary_ids, n_rows, n_cols, min_fovs, ext_range):
    rows = [r for r, c in grid_indices.values()]
    cols = [c for r, c in grid_indices.values()]
    for r0 in range(min(rows), max(rows) - n_rows + 2):
        for c0 in range(min(cols), max(cols) - n_cols + 2):
            block = [fid for fid, (r, c) in grid_indices.items()
                     if r0 <= r < r0 + n_rows and c0 <= c < c0 + n_cols]
            if len(block) < min_fovs:
                continue
            n_ext = sum(1 for f in block if f in boundary_ids)
            if ext_range[0] <= n_ext <= ext_range[1]:
                return r0, c0, sorted(block)
    raise RuntimeError("No block satisfies the given constraints -- widen BLOCK_GRID_ROWS/COLS, "
                        "MIN_BLOCK_FOVS, or EXT_COUNT_RANGE.")

block_r0, block_c0, block_fov_ids = pick_compact_block(
    grid_indices, boundary_fov_ids, BLOCK_GRID_ROWS, BLOCK_GRID_COLS, MIN_BLOCK_FOVS, EXT_COUNT_RANGE,
)
block_n_ext = sum(1 for f in block_fov_ids if f in boundary_fov_ids)
print(f"Block: rows [{block_r0}, {block_r0 + BLOCK_GRID_ROWS}), cols [{block_c0}, {block_c0 + BLOCK_GRID_COLS})")
print(f"{len(block_fov_ids)} real FOV(s), {block_n_ext} boundary FOV(s): {block_fov_ids}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
for fov_id, (x, y) in positions.items():
    in_block = fov_id in block_fov_ids
    ax.add_patch(mpatches.Rectangle(
        (x - half, y - half), 2 * half, 2 * half,
        lw=0.3, edgecolor="tab:blue" if in_block else "0.6",
        facecolor="tab:blue" if in_block else "0.6",
        alpha=0.6 if in_block else 0.1,
    ))
ax.plot([], [], "s", color="tab:blue", alpha=0.6, ms=8, label=f"section 7 block ({len(block_fov_ids)} FOVs)")
ax.invert_yaxis(); ax.axis("equal")
ax.set_title(f"Representative block location -- {SAMPLE_NAME}", fontsize=PLOT_TITLE_FONTSIZE)
ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
ax.legend(fontsize=PLOT_LEGEND_FONTSIZE, loc="best")
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.block_location.png", dpi=150)
plt.show()

## 8 -- Per-FOV elevation matrix `M` + downsampled DAPI z-stack

For every FOV in the block, and every z-plane: FFC-correct, downsample,
threshold (`>= THRESHOLD`), and overwrite `M[i, j] = z_um` wherever the
downsampled pixel is foreground -- iterating z ascending, `M` ends up
holding each pixel's *topmost* foreground z (in µm). `M` starts at 0 (no
sentinel needed: this round's z values start at 0.5 µm, never 0, so 0
unambiguously means "never foreground at any z"). Both `M` and the
downsampled corrected z-stack (reused by the GIF in section 10) are cached
per FOV -- NOTEBOOK_GUIDELINES.md #2/#3, skip-if-cached checked individually
per FOV.

In [ ]:
elevation_dir  = CACHE_DIR / "elevation"
downsampled_dir = CACHE_DIR / "downsampled_stack"
elevation_dir.mkdir(parents=True, exist_ok=True)
downsampled_dir.mkdir(parents=True, exist_ok=True)

def elevation_path(fov_id):
    return elevation_dir / f"fov_{fov_id:04d}.npy"

def downsampled_path(fov_id):
    return downsampled_dir / f"fov_{fov_id:04d}.npz"

def compute_fov_elevation(fpath, z_frame_indices, z_um_values, ffc_field, threshold, factor):
    M = None
    ds_stack = []
    for idx, z_um in zip(z_frame_indices, z_um_values):
        frame = read_image_frames(fpath, [int(idx)])[0]
        ds = ffc_correct_and_downsample(frame, ffc_field, factor).astype(np.float32)
        if M is None:
            M = np.zeros(ds.shape, dtype=np.float32)
        M[ds >= threshold] = z_um   # ascending z -> ends as the topmost foreground z
        ds_stack.append(ds)
    return M, np.stack(ds_stack, axis=0)

to_compute = [f for f in block_fov_ids if not (elevation_path(f).exists() and downsampled_path(f).exists())]
print(f"{len(block_fov_ids) - len(to_compute)} / {len(block_fov_ids)} FOV(s) already cached; "
      f"computing {len(to_compute)} more ({len(z_frame_indices)} z-plane(s) each).")

if to_compute:
    reporter = ProgressReporter(total=len(to_compute), label="Computing per-FOV elevation matrices")
    for fov_id in reporter.wrap(to_compute):
        fpath = round_info.fov_files[fov_id][0]
        M, ds_stack = compute_fov_elevation(fpath, z_frame_indices, z_um_values, ffc_field, THRESHOLD, DOWNSAMPLE_FACTOR)
        np.save(elevation_path(fov_id), M)
        np.savez_compressed(downsampled_path(fov_id), stack=ds_stack, z_um=np.asarray(z_um_values, dtype=np.float32))

elevation_matrices = {f: np.load(elevation_path(f)) for f in block_fov_ids}
print(f"Elevation matrix shape (per FOV): {next(iter(elevation_matrices.values())).shape}")

In [ ]:
example_ids = block_fov_ids[:6]
fig, axes = plt.subplots(1, len(example_ids), figsize=(3 * len(example_ids), 3.2))
vmax = max(float(elevation_matrices[f].max()) for f in example_ids)
for ax, fov_id in zip(np.atleast_1d(axes), example_ids):
    im = ax.imshow(elevation_matrices[fov_id], cmap="viridis", vmin=0, vmax=vmax)
    ax.set_title(f"FOV {fov_id}", fontsize=PLOT_TITLE_FONTSIZE)
    ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
cbar = fig.colorbar(im, ax=np.atleast_1d(axes).tolist(), shrink=0.8, label="elevation (um)")
cbar.ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
fig.suptitle("Example per-FOV elevation matrices (pre-crop)", fontsize=PLOT_TITLE_FONTSIZE)
fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.example_fov_elevation.png", dpi=150)
plt.show()

## 9 -- Stitch: crop overlap + assemble the block's elevation heatmap

Each FOV's downsampled `M` is centrally cropped to its non-overlap
footprint (`analysis.ffc.compute_mosaic_crop_px`, the same overlap-crop
convention `create_mosaic_ffc` already uses for round mosaics, converted
from raw to downsampled pixels), then placed into a grid-indexed canvas
sized `(block_rows * crop, block_cols * crop)` -- missing grid cells (no
real FOV, e.g. a hole) are left as `NaN`.

In [ ]:
crop_px_raw  = compute_mosaic_crop_px(config)
crop_px      = crop_px_raw // DOWNSAMPLE_FACTOR
ds_size      = image_size_px // DOWNSAMPLE_FACTOR
tile_size    = ds_size - 2 * crop_px
print(f"Raw overlap crop: {crop_px_raw} px/side -> downsampled: {crop_px} px/side -> tile size {tile_size}x{tile_size}")

def center_crop(arr, crop_px):
    if crop_px == 0:
        return arr
    return arr[crop_px:-crop_px, crop_px:-crop_px]

def stitch_by_grid(tiles, grid_indices, r0, c0, n_rows, n_cols, crop_px, fill=np.nan):
    cropped = {f: center_crop(t, crop_px) for f, t in tiles.items()}
    tile_h, tile_w = next(iter(cropped.values())).shape
    canvas = np.full((n_rows * tile_h, n_cols * tile_w), fill, dtype=np.float32)
    for fov_id, tile in cropped.items():
        r, c = grid_indices[fov_id]
        rr, cc = r - r0, c - c0
        canvas[rr * tile_h:(rr + 1) * tile_h, cc * tile_w:(cc + 1) * tile_w] = tile
    return canvas

elevation_heatmap = stitch_by_grid(
    elevation_matrices, grid_indices, block_r0, block_c0, BLOCK_GRID_ROWS, BLOCK_GRID_COLS, crop_px,
)
np.save(CACHE_DIR / "elevation_heatmap_block.npy", elevation_heatmap)
print(f"Stitched elevation heatmap shape: {elevation_heatmap.shape}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
cmap = plt.cm.viridis.copy(); cmap.set_bad("0.85")
im = ax.imshow(np.ma.masked_invalid(elevation_heatmap), cmap=cmap)
ax.set_title(f"Tissue elevation heatmap -- {len(block_fov_ids)}-FOV block", fontsize=PLOT_TITLE_FONTSIZE)
ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
cbar = fig.colorbar(im, ax=ax, label="elevation (um)")
cbar.ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.elevation_heatmap.png", dpi=150)
plt.show()

## 10 -- Z-sweep GIF (downsampled, FFC-corrected DAPI mosaic)

Same block, same crop/stitch function as section 9, applied per z-step to
the raw (non-elevation) downsampled intensity stacks cached in section 8 --
one PNG per z-step, assembled into a GIF (same idiom as
`misc/measure_tissue_thickness_test.ipynb` section 24: shared intensity
scale across every frame, so brightness changes reflect real signal
fading, not per-frame auto-contrast).

In [ ]:
downsampled_stacks = {f: np.load(downsampled_path(f))["stack"] for f in block_fov_ids}
n_z = next(iter(downsampled_stacks.values())).shape[0]
gif_z_positions = list(range(0, n_z, GIF_Z_STRIDE))

pooled_pixels = np.concatenate([stack[z].ravel() for stack in downsampled_stacks.values() for z in gif_z_positions])
vmin_g, vmax_g = np.percentile(pooled_pixels, [1.0, 99.0])
print(f"Shared GIF display scale (p1-p99): [{vmin_g:.0f}, {vmax_g:.0f}]")

def to_uint8(arr, vmin, vmax):
    scaled = (arr.astype(np.float64) - vmin) / max(vmax - vmin, 1e-9) * 255
    return np.clip(scaled, 0, 255).astype(np.uint8)

pil_frames = []
reporter = ProgressReporter(total=len(gif_z_positions), label="Assembling GIF frames")
for z_pos in reporter.wrap(gif_z_positions):
    tiles = {f: to_uint8(stack[z_pos], vmin_g, vmax_g) for f, stack in downsampled_stacks.items()}
    canvas = stitch_by_grid(tiles, grid_indices, block_r0, block_c0, BLOCK_GRID_ROWS, BLOCK_GRID_COLS, crop_px, fill=0)
    img = Image.fromarray(canvas.astype(np.uint8), mode="L")
    scale = GIF_DOWNSCALE_WIDTH_PX / img.width
    img = img.resize((GIF_DOWNSCALE_WIDTH_PX, max(1, int(img.height * scale))))
    draw = ImageDraw.Draw(img)
    font = ImageFont.load_default(size=max(14, GIF_DOWNSCALE_WIDTH_PX // 40))
    draw.text((8, 8), f"z = {z_um_values[z_pos]:.1f} um", fill=255, font=font)
    pil_frames.append(img)

gif_path = FIGURES_DIR / f"{NOTEBOOK_NAME}_z_sweep_block.gif"
pil_frames[0].save(gif_path, save_all=True, append_images=pil_frames[1:],
                    duration=GIF_FRAME_DURATION_MS, loop=0)
print(f"Saved: {gif_path}  ({len(pil_frames)} frame(s))")

## Summary

- Boundary (exterior-grid) FOVs identified over the real, full 1138-FOV
  grid: see section 3-4.
- FFC field + FFC-corrected/downsampled-space threshold estimated from
  those boundary FOVs' real data: sections 5-6.
- Per-FOV elevation matrices, crop+stitch, and the z-sweep GIF validated on
  one real 5x5 representative block (straddling a real tissue edge):
  sections 7-10.
- Figures: `{FIGURES_DIR}`. Cache (elevation matrices, downsampled stacks,
  FFC field, stitched heatmap): `{CACHE_DIR}`.

**Follow-up for another session** (per this investigation's own scope):
promote `compute_fov_elevation`/`center_crop`/`stitch_by_grid` into
`MERci` proper (`analysis/` -- alongside `ffc.py`/`round.py`), and run
section 8 over the real full 1138-FOV grid via a SLURM array job (same
pattern as `misc/measure_tissue_thickness_test.ipynb` sections 14/23/24),
then wire the result into `after_imaging/08_measure_tissue_thickness.ipynb`.